In [ ]:
import sys
from pathlib import Path

# --- Colab/Local Setup ---
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab. Setting up environment...")
    from google.colab import drive
    drive.mount('/content/drive')
    project_path = "/content/drive/MyDrive/data"
    !ln -sfn {project_path} /project
    BASE_PATH = Path("/project")
    DATASET_PATH = BASE_PATH / "dataset_lma_effort.pkl"
    REPO_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path("..").resolve()
    sys.path.append(str(REPO_ROOT))
    %pip install -q bvhio
else:
    print("Not in Colab. Assuming local setup.")
    BASE_PATH = Path("./")
    REPO_ROOT = Path("..").resolve()
    sys.path.insert(0, str(REPO_ROOT))
    DATASET_PATH = REPO_ROOT / "datasets" / "lma_effort.pkl"


In [ ]:
from pathlib import Path

from source.dataset import build_lma_effort_dataset

LMA_EFFORT_DATASET_ROOT = Path('/project/extracted_fingerless')  # update for your environment
DATASET_PATH = LMA_EFFORT_DATASET_ROOT.parent / 'lma_effort.pkl'

dataset = build_lma_effort_dataset(
    dataset_root=LMA_EFFORT_DATASET_ROOT,
    save_path=DATASET_PATH,
    device='cpu',
    sequence_length=50,
    max_frame_diff=1,
)
print(dataset.laban_limits)
del dataset


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm

from source.dataset import MotionDataset
from source.forward_kinematics import ForwardKinematics, sixd_to_matrix
from source.rotations import rotation_6d_to_matrix
from source.synthesizer import Synthesizer


class TrainingConfig:
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    DATASET_PATH = DATASET_PATH
    SAVE_SYNTHESIZER_PATH = BASE_PATH / "synthesizer.pth"
    BATCH_SIZE = 64
    LEARNING_RATE = 1e-4
    NUM_EPOCHS = 300
    VAL_SPLIT = 0.1

    # Manuscript Section 3.3.2 loss weights.
    LAMBDA_IK = 40.0
    LAMBDA_SYNTH = 1.0
    LAMBDA_V = 1.0
    LAMBDA_H = 1.0
    LAMBDA_P = 1.0
    LAMBDA_R = 2.0

    # Architecture per Section 3.1: single-layer LSTM with hidden 128,
    # followed by three 512-node fully connected layers.
    SYNTH_HIDDEN_DIM = 128
    SYNTH_FC_DIM = 512
    SYNTH_LAYERS = 1

    # Style-descriptor noise / randomization (Section 3.3.2).
    NOISE_VH = 0.2
    # lambda for the R descriptor std term (Section 3.2.4).
    R_LAMBDA = 1.0


config = TrainingConfig()
print(f"Using device: {config.DEVICE}")
print(f"Synthesizer model will be saved to: {config.SAVE_SYNTHESIZER_PATH}")

print(f"Loading pre-processed dataset from {config.DATASET_PATH}...")
if not config.DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {config.DATASET_PATH}.")

motion_dataset = MotionDataset.load(config.DATASET_PATH, device='cpu')

NUM_JOINTS = motion_dataset.num_joints
SITE_IDS = motion_dataset.site_ids
ANGLES_DIM_PER_JOINT = 6
POSITIONS_DIM_PER_SITE = 3
ANGLES_DIM = NUM_JOINTS * ANGLES_DIM_PER_JOINT
POSITIONS_DIM = len(SITE_IDS) * POSITIONS_DIM_PER_SITE


class SynthesizerDataset(Dataset):
    def __init__(self, source: MotionDataset):
        self.source = source

    def __len__(self) -> int:
        return len(self.source)

    def __getitem__(self, index: int):
        return self.source[index]


full_dataset = SynthesizerDataset(motion_dataset)
train_size = int(len(full_dataset) * (1 - config.VAL_SPLIT))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, pin_memory=True)

print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

fk_engine: ForwardKinematics = motion_dataset.converter
fk_engine.device = config.DEVICE
fk_engine.offsets = fk_engine.offsets.to(config.DEVICE)

synthesizer = Synthesizer(
    angles_dim=ANGLES_DIM,
    positions_dim=POSITIONS_DIM,
    conditions_dim=4,
    hidden_dim=config.SYNTH_HIDDEN_DIM,
    fc_dim=config.SYNTH_FC_DIM,
    num_layers=config.SYNTH_LAYERS,
    dropout=0.0,
).to(config.DEVICE)


class GeodesicLoss(nn.Module):
    """Geodesic distance between two batches of rotations given as 6D vectors."""

    def __init__(self, epsilon: float = 1e-7, reduction: str = 'mean'):
        super().__init__()
        self.epsilon = epsilon
        self.reduction = reduction

    def forward(self, pred_6d: torch.Tensor, gt_6d: torch.Tensor) -> torch.Tensor:
        pred_R = rotation_6d_to_matrix(pred_6d)
        gt_R = rotation_6d_to_matrix(gt_6d)
        error_R = torch.matmul(pred_R, gt_R.transpose(-1, -2))
        trace = torch.diagonal(error_R, offset=0, dim1=-2, dim2=-1).sum(-1)
        ratio = torch.clamp((trace - 1) / 2, -1.0 + self.epsilon, 1.0 - self.epsilon)
        angle = torch.acos(ratio)
        if self.reduction == 'mean':
            return angle.mean()
        if self.reduction == 'sum':
            return angle.sum()
        if self.reduction == 'none':
            return angle
        raise ValueError(f"Unknown reduction: {self.reduction}")


# Manuscript Section 3.3.2: L_synth is geodesic; L_IK is MSE between
# mean-centered site positions; L_V, L_H, L_P, L_R are L1 losses between
# computed and target style descriptor values.
angle_loss_fn = GeodesicLoss(reduction='mean')
ik_loss_fn = nn.MSELoss()
style_loss_fn = nn.L1Loss()

optimizer = optim.AdamW(synthesizer.parameters(), lr=config.LEARNING_RATE)
print(f"Synthesizer has {sum(p.numel() for p in synthesizer.parameters() if p.requires_grad):,} trainable parameters.")

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.2, patience=5, verbose=True,
)


def normalize(tensor, low, high):
    return (tensor - low) / (high - low)


def apply_training_noise(conditions: torch.Tensor, noise_vh: float) -> torch.Tensor:
    """+/- noise on V and H, uniform R randomization (Section 3.3.2)."""
    v_noise = (torch.rand_like(conditions[:, 0]) * 2.0 - 1.0) * noise_vh
    h_noise = (torch.rand_like(conditions[:, 1]) * 2.0 - 1.0) * noise_vh
    conditions[:, 0] = conditions[:, 0] + v_noise
    conditions[:, 1] = conditions[:, 1] + h_noise
    conditions[:, 3] = torch.rand_like(conditions[:, 3])
    return conditions


def run_epoch(synthesizer, fk_engine, loader, optimizer, config, site_ids, is_training):
    if is_training:
        synthesizer.train()
        desc = "Training"
    else:
        synthesizer.eval()
        desc = "Validating"

    sums = [0.0] * 7
    pbar = tqdm(loader, desc=desc)
    for batch in pbar:
        for key in batch:
            if isinstance(batch[key], torch.Tensor):
                batch[key] = batch[key].to(config.DEVICE)

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):
            positions_sites_mz = batch['positions_sites'] - batch['positions_sites'].mean(dim=-2, keepdim=True)
            batch_size, seq_len, _, _ = positions_sites_mz.shape

            conditions = torch.stack([
                batch['laban_v'], batch['laban_h'],
                batch['laban_p'], batch['laban_r'],
            ], dim=-1).clone()
            if is_training:
                conditions = apply_training_noise(conditions, config.NOISE_VH)

            pred_flat = synthesizer(
                dense_trajectory=positions_sites_mz.view(batch_size, seq_len, POSITIONS_DIM),
                conditions=conditions,
            )
            pred_6d = pred_flat.view(batch_size, seq_len, NUM_JOINTS, ANGLES_DIM_PER_JOINT)
            pred_matrix = sixd_to_matrix(pred_6d)
            gt_matrix = sixd_to_matrix(batch['angles_6d'])

            L_synth = angle_loss_fn(pred_6d, batch['angles_6d'])

            recon_positions_all, _ = fk_engine.compute(
                rotations=pred_matrix,
                root_translations=torch.zeros(batch_size, seq_len, 3, device=config.DEVICE),
            )
            recon_sites = recon_positions_all[..., site_ids, :]
            recon_sites_mz = recon_sites - recon_sites.mean(dim=-2, keepdim=True)
            L_IK = ik_loss_fn(
                recon_sites_mz,
                positions_sites_mz.view(batch_size, seq_len, len(site_ids), POSITIONS_DIM_PER_SITE),
            )

            laban_v = normalize(motion_dataset.laban.vertical(recon_positions_all), *motion_dataset.laban_limits['v'])
            L_V = style_loss_fn(laban_v, conditions[:, 0])

            laban_h = normalize(motion_dataset.laban.horizontal(recon_positions_all), *motion_dataset.laban_limits['h'])
            L_H = style_loss_fn(laban_h, conditions[:, 1])

            # P averages speed over every joint (Section 3.2.3).
            laban_p = normalize(motion_dataset.laban.pace(recon_positions_all), *motion_dataset.laban_limits['p'])
            L_P = style_loss_fn(laban_p, conditions[:, 2])

            # R: geodesic between predicted and gt rotations plus joint-wise std
            # (Section 3.2.4). Normalized using fixed [0, pi] limits.
            laban_r = motion_dataset.laban.regularity(pred_matrix, gt_matrix, lambda_=config.R_LAMBDA)
            laban_r = normalize(laban_r, *motion_dataset.laban_limits['r'])
            L_R = style_loss_fn(laban_r, conditions[:, 3])

            total = (
                config.LAMBDA_IK * L_IK
                + config.LAMBDA_SYNTH * L_synth
                + config.LAMBDA_V * L_V
                + config.LAMBDA_H * L_H
                + config.LAMBDA_P * L_P
                + config.LAMBDA_R * L_R
            )

        if is_training:
            total.backward()
            optimizer.step()

        for j, value in enumerate([total, L_IK, L_synth, L_V, L_H, L_P, L_R]):
            sums[j] += value.item()

        pbar.set_postfix({
            'Total': f"{total.item():.4f}",
            'IK': f"{L_IK.item():.6f}",
            'Sim': f"{L_synth.item():.4f}",
            'V': f"{L_V.item():.4f}",
            'H': f"{L_H.item():.4f}",
            'P': f"{L_P.item():.4f}",
            'R': f"{L_R.item():.4f}",
        })

    n = len(loader)
    return tuple(s / n for s in sums)


best_val_loss = float('inf')
for epoch in range(config.NUM_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{config.NUM_EPOCHS} ---")
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr:.1e}")

    tr_total, tr_ik, tr_sim, tr_v, tr_h, tr_p, tr_r = run_epoch(
        synthesizer, fk_engine, train_loader, optimizer, config, SITE_IDS, is_training=True,
    )
    print(f"Epoch {epoch+1} | Train | Total: {tr_total:.4f} | IK: {tr_ik:.6f} | Sim: {tr_sim:.4f} | V: {tr_v:.4f} | H: {tr_h:.4f} | P: {tr_p:.4f} | R: {tr_r:.4f}")

    va_total, va_ik, va_sim, va_v, va_h, va_p, va_r = run_epoch(
        synthesizer, fk_engine, val_loader, optimizer, config, SITE_IDS, is_training=False,
    )
    print(f"Epoch {epoch+1} | Valid | Total: {va_total:.4f} | IK: {va_ik:.6f} | Sim: {va_sim:.4f} | V: {va_v:.4f} | H: {va_h:.4f} | P: {va_p:.4f} | R: {va_r:.4f}")

    scheduler.step(va_total)

    if va_total < best_val_loss:
        best_val_loss = va_total
        print(f"New best validation loss. Saving model to {config.SAVE_SYNTHESIZER_PATH}")
        torch.save(synthesizer.state_dict(), config.SAVE_SYNTHESIZER_PATH)

print("\nTraining complete.")
